In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# -----------------------
# SPARK SESSION
# -----------------------
spark = (
    SparkSession.builder
    .appName("Spotify Silver ETL")
    .config("spark.driver.memory", "16g")
    .config("spark.executor.memory", "16g")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", 200)
    .config("spark.sql.adaptive.enabled", "true")
    .enableHiveSupport()
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/05 22:47:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
BRONZE_BASE = "hdfs://localhost:9000/data_lake/bronze"
SILVER_BASE = "hdfs://localhost:9000/data_lake/silver"
GOLD_BASE = "hdfs://localhost:9000/data_lake/gold"

# Bronze

In [3]:
artists = spark.read.parquet(f"{BRONZE_BASE}/artists/batch_id=3")
albums = spark.read.parquet(f"{BRONZE_BASE}/albums/batch_id=3")
tracks = spark.read.parquet(f"{BRONZE_BASE}/tracks/batch_id=3")

In [3]:
artist_genres = spark.read.parquet(f"{BRONZE_BASE}/artist_genres")
track_artists = spark.read.parquet(f"{BRONZE_BASE}/track_artists")
artist_albums = spark.read.parquet(f"{BRONZE_BASE}/artist_albums")
available_markets = spark.read.parquet(f"{BRONZE_BASE}/available_markets")

print("✅ Bronze tables loaded (using natural partitioning)")

✅ Bronze tables loaded (using natural partitioning)


In [4]:
spark.read.parquet(f"{BRONZE_BASE}/artists/batch_id=3").limit(10).show(10, truncate=False)

+--------+----------------------+-------------+------------------+---------------+----------+-----------------------+--------+
|rowid   |id                    |fetched_at   |name              |followers_total|popularity|ingestion_ts           |batch_id|
+--------+----------------------+-------------+------------------+---------------+----------+-----------------------+--------+
|13764608|5Bvt7Fcg2IWsWqRbEvZKIa|1756944000000|KYDIN             |39             |8         |2026-01-30 08:52:25.824|3       |
|13764609|0ZTVIw09zsjiz1oR1xJQZp|1756944000000|Pearl of Silence  |2              |18        |2026-01-30 08:52:25.824|3       |
|13764610|4hf1DKkNOMWLDRplNUuGxZ|1756944000000|Takuto            |0              |0         |2026-01-30 08:52:25.824|3       |
|13764611|44DFtMlZxXIfQBnoLUaoms|1756944000000|Sienipilvi        |132            |0         |2026-01-30 08:52:25.824|3       |
|13764612|77M8cJPnXvd45hyyE1Hdo9|1756944000000|edvin.            |0              |0         |2026-01-30 08:52:2

In [5]:
spark.read.parquet(f"{BRONZE_BASE}/albums/batch_id=3").limit(10).show(10, truncate=False)

+--------+----------------------+-------------+-------------------------------------------+----------+-----------------------+---------------+--------------------------------+--------------------------------+---------------------------+----------+------------+----------------------+------------+-----------------+-----------------------+--------+
|rowid   |id                    |fetched_at   |name                                       |album_type|available_markets_rowid|external_id_upc|copyright_c                     |copyright_p                     |label                      |popularity|release_date|release_date_precision|total_tracks|external_id_amgid|ingestion_ts           |batch_id|
+--------+----------------------+-------------+-------------------------------------------+----------+-----------------------+---------------+--------------------------------+--------------------------------+---------------------------+----------+------------+----------------------+------------+--------

In [6]:
spark.read.parquet(f"{BRONZE_BASE}/tracks/batch_id=3").limit(10).show(10, truncate=False)

+---------+----------------------+-------------+----------------------------+-----------------------------------------------------------------------------------------------------------+-----------+------------+----------------+----------+-----------------------+-----------+-----------+--------+-----------------------+--------+
|rowid    |id                    |fetched_at   |name                        |preview_url                                                                                                |album_rowid|track_number|external_id_isrc|popularity|available_markets_rowid|disc_number|duration_ms|explicit|ingestion_ts           |batch_id|
+---------+----------------------+-------------+----------------------------+-----------------------------------------------------------------------------------------------------------+-----------+------------+----------------+----------+-----------------------+-----------+-----------+--------+-----------------------+--------+
|238757888|5q

In [7]:
spark.read.parquet(f"{BRONZE_BASE}/artist_genres").limit(10).show(10, truncate=False)

+------------+-------------------+-----------------------+--------+
|artist_rowid|genre              |ingestion_ts           |batch_id|
+------------+-------------------+-----------------------+--------+
|3           |luk thung          |2026-01-30 08:10:52.362|-1      |
|4           |trance             |2026-01-30 08:10:52.362|-1      |
|11          |celtic             |2026-01-30 08:10:52.362|-1      |
|11          |traditional music  |2026-01-30 08:10:52.362|-1      |
|11          |folk               |2026-01-30 08:10:52.362|-1      |
|27          |tribal house       |2026-01-30 08:10:52.362|-1      |
|31          |chinese hip hop    |2026-01-30 08:10:52.362|-1      |
|37          |afropop            |2026-01-30 08:10:52.362|-1      |
|38          |southern thai music|2026-01-30 08:10:52.362|-1      |
|51          |lullaby            |2026-01-30 08:10:52.362|-1      |
+------------+-------------------+-----------------------+--------+



In [8]:
spark.read.parquet(f"{BRONZE_BASE}/track_artists").limit(10).show(10, truncate=False)

+-----------+------------+-----------------------+--------+
|track_rowid|artist_rowid|ingestion_ts           |batch_id|
+-----------+------------+-----------------------+--------+
|15911746   |2600240     |2026-01-30 08:10:52.362|-1      |
|15911747   |2546254     |2026-01-30 08:10:52.362|-1      |
|15911748   |1385521     |2026-01-30 08:10:52.362|-1      |
|15911749   |8782488     |2026-01-30 08:10:52.362|-1      |
|15911750   |4051901     |2026-01-30 08:10:52.362|-1      |
|15911750   |573858      |2026-01-30 08:10:52.362|-1      |
|15911750   |1753422     |2026-01-30 08:10:52.362|-1      |
|15911751   |8954593     |2026-01-30 08:10:52.362|-1      |
|15911752   |4565211     |2026-01-30 08:10:52.362|-1      |
|15911753   |4461991     |2026-01-30 08:10:52.362|-1      |
+-----------+------------+-----------------------+--------+



In [9]:
spark.read.parquet(f"{BRONZE_BASE}/artist_albums").limit(10).show(10, truncate=False)

+------------+-----------+-------------+----------------------+--------------+-----------------------+--------+
|artist_rowid|album_rowid|is_appears_on|is_implicit_appears_on|index_in_album|ingestion_ts           |batch_id|
+------------+-----------+-------------+----------------------+--------------+-----------------------+--------+
|12337833    |46200700   |0            |0                     |0             |2026-01-30 08:10:52.362|-1      |
|12337833    |46200701   |0            |0                     |0             |2026-01-30 08:10:52.362|-1      |
|12337833    |46200702   |0            |0                     |0             |2026-01-30 08:10:52.362|-1      |
|12337833    |46200703   |0            |0                     |0             |2026-01-30 08:10:52.362|-1      |
|12337836    |46200704   |0            |0                     |0             |2026-01-30 08:10:52.362|-1      |
|12337836    |46200705   |0            |0                     |0             |2026-01-30 08:10:52.362|-1

In [10]:
spark.read.parquet(f"{BRONZE_BASE}/available_markets").limit(10).show(10, truncate=False)

+-----+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------+--------+
|rowid|available_markets                                                                                                                                                                                                                                                                                                                                                                                            

# Silver

## 1. artists_clean

In [ ]:
artists_clean = (
    artists
    # 1. TRANSFORM FIRST - this reduces data size
    .dropDuplicates(["rowid"])
    .withColumn("fetched_ts", to_timestamp(col("fetched_at") / 1000))
    .withColumn("followers_total", coalesce(col("followers_total"), lit(0)))
    .withColumn("popularity", coalesce(col("popularity"), lit(0)))
    .select(
        col("rowid").alias("artist_rowid"),
        col("id").alias("artist_id"),
        "name",
        "followers_total",
        "popularity",
        "fetched_ts"
    )
    # 2. REPARTITION AFTER transformation (smaller data now)
    # 617 MB / 77 MB per partition = ~8 partitions
    .repartition(8, "artist_rowid")  # Partition by PK for future joins
)

# 3. WRITE
artists_clean.write.mode("overwrite").parquet(f"{SILVER_BASE}/artists_clean")
print("✅ artists_clean written (8 partitions)")

# 4. VERIFY
print("\n📊 Sample data:")
spark.read.parquet(f"{SILVER_BASE}/artists_clean").show(5, truncate=False)

print(f"📈 Partition count: {spark.read.parquet(f'{SILVER_BASE}/artists_clean').rdd.getNumPartitions()}")

✅ artists_clean written (8 partitions)

📊 Sample data:
+------------+----------------------+--------------------------------+---------------+----------+-------------------+
|artist_rowid|artist_id             |name                            |followers_total|popularity|fetched_ts         |
+------------+----------------------+--------------------------------+---------------+----------+-------------------+
|13476505    |1XYtaVJy7P2iJdiHIXRZ4l|Lepton                          |0              |0         |2025-08-21 05:30:00|
|13476536    |2I24pIA60G500vEXW7yibT|アルトゥール・ルービンシュタイン|4              |0         |2025-08-21 05:30:00|
|13476618    |1dfNbqHxQZEoJOD4uRcBW2|Naofat Jahan Naba               |1              |0         |2025-08-21 05:30:00|
|13476683    |2COIV8B80KHS5gwRfy6DAM|Brett Carlisle                  |4              |0         |2025-08-21 05:30:00|
|13476694    |6FqsvH6L9Aa1Vc5R1OSBot|WORK                            |1              |0         |2025-08-21 05:30:00|
+------------+---

## 2. albums_clean

In [12]:
albums.select(year("release_date")).distinct().orderBy("year(release_date)").show()

+------------------+
|year(release_date)|
+------------------+
|                 0|
|                 1|
|                 2|
|                 6|
|                13|
|                16|
|                17|
|                20|
|                23|
|               202|
|               208|
|               225|
|               571|
|               667|
|              1000|
|              1001|
|              1013|
|              1054|
|              1200|
|              1207|
+------------------+
only showing top 20 rows



In [13]:
albums.filter(col("release_date_precision") == "day") \
      .select("release_date") \
      .distinct() \
      .show(10)

+------------+
|release_date|
+------------+
|  2023-05-01|
|  2019-08-08|
|  2024-01-19|
|  2019-08-23|
|  2021-11-03|
|  2014-02-22|
|  2024-10-24|
|  2002-06-21|
|  2019-08-22|
|  1993-06-22|
+------------+
only showing top 10 rows



In [14]:
albums.filter(col("release_date_precision") == "month") \
      .select("release_date") \
      .distinct() \
      .show(10)

+------------+
|release_date|
+------------+
|     1981-10|
|     2007-05|
|     1983-03|
|     1957-06|
|     1980-09|
|     2001-05|
|     1970-08|
|     1969-04|
|     1987-11|
|     1967-12|
+------------+
only showing top 10 rows



In [15]:
albums.filter(col("release_date_precision") == "year") \
      .select("release_date") \
      .distinct() \
      .show(10)

+------------+
|release_date|
+------------+
|        1953|
|        1957|
|        1897|
|        1987|
|        1956|
|        2016|
|        1936|
|        2012|
|        2020|
|        1958|
+------------+
only showing top 10 rows



In [16]:
albums_clean = (
    albums
    # 1. TRANSFORM FIRST
    .dropDuplicates(["rowid"])
    
    # Normalize release_date based on precision
    .withColumn(
        "release_date_standardized",
        when(col("release_date_precision") == "year",
             concat(col("release_date"), lit("-01-01")))
        .when(col("release_date_precision") == "month",
             concat(col("release_date"), lit("-01")))
        .otherwise(col("release_date"))
    )
    
    # Safe date conversion
    .withColumn("release_date_parsed", to_date("release_date_standardized"))
    
    # Filter corrupted / ancient Spotify records
    .filter(
        col("release_date_parsed").isNotNull() &
        (year("release_date_parsed") >= 1900)
    )
    
    .withColumn("release_year", year("release_date_parsed"))
    .withColumn("fetched_ts", to_timestamp(col("fetched_at") / 1000))
    .withColumn("popularity", coalesce(col("popularity"), lit(0)))
    .withColumn("total_tracks", coalesce(col("total_tracks"), lit(0)))
    
    .select(
        col("rowid").alias("album_rowid"),
        col("id").alias("album_id"),
        "name",
        "album_type",
        "popularity",
        "release_date_parsed",
        "release_year",
        "total_tracks",
        "available_markets_rowid",
        "fetched_ts"
    )
    # 2. REPARTITION AFTER transformation
    # 4.4 GB / 137 MB per partition = ~32 partitions
    .repartition(32, "album_rowid")
)

# 3. WRITE
albums_clean.write.mode("overwrite").parquet(f"{SILVER_BASE}/albums_clean")
print("✅ albums_clean written (32 partitions)")

# 4. VERIFY
print("\n📊 Sample data:")
spark.read.parquet(f"{SILVER_BASE}/albums_clean").show(5, truncate=False)

print(f"📈 Partition count: {spark.read.parquet(f'{SILVER_BASE}/albums_clean').rdd.getNumPartitions()}")


✅ albums_clean written (32 partitions)

📊 Sample data:
+-----------+----------------------+--------------------------------------+-----------+----------+-------------------+------------+------------+-----------------------+-------------------+
|album_rowid|album_id              |name                                  |album_type |popularity|release_date_parsed|release_year|total_tracks|available_markets_rowid|fetched_ts         |
+-----------+----------------------+--------------------------------------+-----------+----------+-------------------+------------+------------+-----------------------+-------------------+
|49982212   |1lTIVcweDrsxBto7TVSMSa|Supply The Work                       |single     |0         |2025-06-26         |2025        |1           |3                      |2025-08-21 05:30:00|
|49982251   |4DYMcNFuQBGRZ2qFeKtDG8|TST                                   |single     |0         |2025-08-02         |2025        |1           |3                      |2025-08-21 05:30:00|


## 3. tracks_clean

In [18]:
tracks_clean = (
    tracks
    # 1. TRANSFORM FIRST
    .dropDuplicates(["rowid"])
    .withColumn("fetched_ts", to_timestamp(col("fetched_at") / 1000))
    .withColumn("duration_seconds", col("duration_ms") / 1000)
    .withColumn(
        "duration_bucket",
        when(col("duration_seconds") < 120, "short")
        .when(col("duration_seconds") < 240, "medium")
        .otherwise("long")
    )
    .withColumn("explicit_flag", col("explicit").cast("boolean"))
    .withColumn("popularity", coalesce(col("popularity"), lit(0)))
    .select(
        col("rowid").alias("track_rowid"),
        col("id").alias("track_id"),
        "name",
        "album_rowid",
        "duration_ms",
        "duration_seconds",
        "duration_bucket",
        "explicit_flag",
        "popularity",
        "available_markets_rowid",
        "fetched_ts"
    )
    # 2. REPARTITION AFTER transformation
    # 22.3 GB / 174 MB per partition = ~128 partitions
    .repartition(128, "track_rowid")
)

In [19]:
# 3. WRITE
tracks_clean.write.mode("overwrite").parquet(f"{SILVER_BASE}/tracks_clean")
print("✅ tracks_clean written (128 partitions)")

# 4. VERIFY
print("\n📊 Sample data:")
spark.read.parquet(f"{SILVER_BASE}/tracks_clean").show(5, truncate=False)

print(f"📈 Partition count: {spark.read.parquet(f'{SILVER_BASE}/tracks_clean').rdd.getNumPartitions()}")


✅ tracks_clean written (128 partitions)

📊 Sample data:
+-----------+----------------------+--------------------------------------------------------------------------+-----------+-----------+----------------+---------------+-------------+----------+-----------------------+-------------------+
|track_rowid|track_id              |name                                                                      |album_rowid|duration_ms|duration_seconds|duration_bucket|explicit_flag|popularity|available_markets_rowid|fetched_ts         |
+-----------+----------------------+--------------------------------------------------------------------------+-----------+-----------+----------------+---------------+-------------+----------+-----------------------+-------------------+
|218161456  |5H6lw40pOvZYZygPtnwFPB|撤                                                                        |49982441   |235000     |235.0           |medium         |false        |0         |1                      |2025-08-21 05:

## CACHE REUSED DATAFRAMES

In [ ]:
print("\n💾 Caching frequently used tables...")

artists_clean = spark.read.parquet(f"{SILVER_BASE}/artists_clean")
albums_clean = spark.read.parquet(f"{SILVER_BASE}/albums_clean")
tracks_clean = spark.read.parquet(f"{SILVER_BASE}/tracks_clean")

# Force caching by triggering an action
artists_clean.count()
albums_clean.count()
tracks_clean.count()

print("✅ Cached: artists_clean, albums_clean, tracks_clean")



💾 Caching frequently used tables...


✅ Cached: artists_clean, albums_clean, tracks_clean


## 4. track_artist_expanded

In [9]:
track_artist_expanded = (
    track_artists
    # 1. DEDUPE FIRST (reduces size before join)
    .dropDuplicates()
    
    # 2. JOIN without pre-repartition
    # Let Spark's AQE (Adaptive Query Execution) handle the shuffle
    .join(artists_clean, "artist_rowid", "left")
    
    .select(
        "track_rowid",
        "artist_rowid",
        "artist_id",
        col("popularity").alias("artist_popularity"),
        "followers_total"
    )
    # 3. SINGLE repartition after join by the key used downstream (track_rowid)
    # This is better than repartitioning twice
    .repartition(32, "track_rowid")
)
print("defined")

defined


In [10]:
# 4. WRITE
track_artist_expanded.write.mode("overwrite").parquet(f"{SILVER_BASE}/track_artist_expanded")
print("✅ track_artist_expanded written (32 partitions)")

26/02/05 18:43:35 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 18:43:35 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 18:43:43 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 18:43:43 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 18:43:43 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 18:43:43 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 18:43:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 18:43:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 18:43:46 WARN RowBasedKeyValueBatch: Calling spill() on

✅ track_artist_expanded written (32 partitions)


In [11]:
# 5. VERIFY
print("\n📊 Sample data (with non-null artist_id):")
spark.read.parquet(f"{SILVER_BASE}/track_artist_expanded").filter(
    col("artist_id").isNotNull()
).show(5, truncate=False)

print(f"📈 Partition count: {spark.read.parquet(f'{SILVER_BASE}/track_artist_expanded').rdd.getNumPartitions()}")


📊 Sample data (with non-null artist_id):
+-----------+------------+----------------------+-----------------+---------------+
|track_rowid|artist_rowid|artist_id             |artist_popularity|followers_total|
+-----------+------------+----------------------+-----------------+---------------+
|220148624  |13477050    |4CGc8jCFmS32rmWpqYCfBJ|0                |3              |
|220150287  |13477441    |1I2qSnLkkv4otDWdR0AkyD|0                |0              |
|251825563  |13477847    |1vnNApG553eZC2dYaoLilP|0                |0              |
|218485111  |13478904    |3bFXGYmUN2Gnf7Yg3HK6zc|0                |0              |
|218485057  |13478904    |3bFXGYmUN2Gnf7Yg3HK6zc|0                |0              |
+-----------+------------+----------------------+-----------------+---------------+
only showing top 5 rows

📈 Partition count: 32


In [ ]:
# TO SHOW NON-NULL VALUES
# track_artist_expanded.filter(col("artist_id").isNotNull()).limit(10).show(10, truncate=False)

## 5. artist_album_expanded

In [12]:
artist_album_expanded = (
    artist_albums
    # 1. DEDUPE
    .dropDuplicates()
    
    # 2. REPARTITION BY JOIN KEY (album_rowid)
    .repartition(16, "album_rowid")
    
    # 3. JOIN with albums_clean (which is cached - fast!)
    .join(albums_clean, "album_rowid", "left")
    
    .select(
        "artist_rowid",
        "album_rowid",
        "release_year",
        col("popularity").alias("album_popularity")
    )
    # 4. FINAL PARTITIONING - keep by album_rowid for future queries
    .repartition(16, "album_rowid")
)

# 5. WRITE
artist_album_expanded.write.mode("overwrite").parquet(f"{SILVER_BASE}/artist_album_expanded")
print("✅ artist_album_expanded written (16 partitions)")

# 6. VERIFY
print("\n📊 Sample data:")
spark.read.parquet(f"{SILVER_BASE}/artist_album_expanded").show(5, truncate=False)

print(f"📈 Partition count: {spark.read.parquet(f'{SILVER_BASE}/artist_album_expanded').rdd.getNumPartitions()}")


26/02/05 18:54:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 18:54:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 18:54:02 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 18:54:02 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 18:54:03 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 18:54:03 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 18:54:03 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 18:54:03 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 18:54:03 WARN RowBasedKeyValueBatch: Calling spill() on

✅ artist_album_expanded written (16 partitions)

📊 Sample data:
+------------+-----------+------------+----------------+
|artist_rowid|album_rowid|release_year|album_popularity|
+------------+-----------+------------+----------------+
|5758083     |155        |NULL        |NULL            |
|6508522     |155        |NULL        |NULL            |
|3715782     |155        |NULL        |NULL            |
|2074035     |155        |NULL        |NULL            |
|5168840     |155        |NULL        |NULL            |
+------------+-----------+------------+----------------+
only showing top 5 rows

📈 Partition count: 16


## 6. artist_genre_expanded

In [5]:
artist_genre_expanded = (
    artist_genres
    # 1. DEDUPE & TRANSFORM
    .dropDuplicates()
    .withColumn("genre", lower(trim(col("genre"))))
    
    # 2. COALESCE (not repartition) - no shuffle needed for tiny data
    # This just combines partitions, doesn't redistribute
    .coalesce(2)
)

# 3. WRITE
artist_genre_expanded.write.mode("overwrite").parquet(f"{SILVER_BASE}/artist_genre_expanded")
print("✅ artist_genre_expanded written (2 partitions)")

# 4. VERIFY
print("\n📊 Sample data:")
spark.read.parquet(f"{SILVER_BASE}/artist_genre_expanded").show(5, truncate=False)

print(f"📈 Partition count: {spark.read.parquet(f'{SILVER_BASE}/artist_genre_expanded').rdd.getNumPartitions()}")


✅ artist_genre_expanded written (2 partitions)

📊 Sample data:
+------------+------------------+-----------------------+--------+
|artist_rowid|genre             |ingestion_ts           |batch_id|
+------------+------------------+-----------------------+--------+
|496         |dark trap         |2026-01-30 08:10:52.362|-1      |
|658         |acid house        |2026-01-30 08:10:52.362|-1      |
|2623        |choral            |2026-01-30 08:10:52.362|-1      |
|5024        |jazz              |2026-01-30 08:10:52.362|-1      |
|6109        |canzone napoletana|2026-01-30 08:10:52.362|-1      |
+------------+------------------+-----------------------+--------+
only showing top 5 rows

📈 Partition count: 4


## 7. market_expanded

In [7]:
# STEP 1: Explode markets (this creates billions of rows!)
markets_exploded = (
    available_markets
    .withColumn("market_code", explode(split(col("available_markets"), ",")))
    .select(
        col("rowid").alias("available_markets_rowid"), 
        "market_code"
    )
    # Repartition by market_code for efficient join
    .repartition(64, "market_code")
)

# STEP 2: Join with tracks_clean (cached - good!)
# We partition tracks by available_markets_rowid before join
tracks_for_markets = (
    tracks_clean
    .select("track_rowid", "available_markets_rowid")
    .repartition(64, "available_markets_rowid")
)

market_expanded = (
    tracks_for_markets
    .join(markets_exploded, "available_markets_rowid", "left")
    .select("track_rowid", "market_code")
    # Repartition by market_code for future market-based queries
    .repartition(64, "market_code")
)

# 3. WRITE
market_expanded.write.mode("overwrite").parquet(f"{SILVER_BASE}/market_expanded")
print("✅ market_expanded written (64 partitions)")

# 4. VERIFY
print("\n📊 Sample data:")
spark.read.parquet(f"{SILVER_BASE}/market_expanded").show(5, truncate=False)

print(f"📈 Partition count: {spark.read.parquet(f'{SILVER_BASE}/market_expanded').rdd.getNumPartitions()}")


✅ market_expanded written (64 partitions)

📊 Sample data:
+-----------+-----------+
|track_rowid|market_code|
+-----------+-----------+
|255882391  |PL         |
|255882391  |ZW         |
|255882391  |BI         |
|218628895  |ZW         |
|218628895  |PL         |
+-----------+-----------+
only showing top 5 rows

📈 Partition count: 62


## CACHE FOR AGGREGATES

In [ ]:
# Load and cache track_artist_expanded for upcoming aggregates

print("\n💾 Loading track_artist_expanded for aggregates...")

track_artist_expanded = spark.read.parquet(f"{SILVER_BASE}/track_artist_expanded")
track_artist_expanded.count()  # Force caching

print("✅ track_artist_expanded cached")


💾 Loading track_artist_expanded for aggregates...


✅ track_artist_expanded cached


## 8. track_artist_aggregates 

In [14]:
track_artist_aggregates = (
    track_artist_expanded
    # 1. GROUP BY (Spark will shuffle based on spark.sql.shuffle.partitions = 200)
    .groupBy("track_rowid")
    .agg(
        countDistinct("artist_rowid").alias("artist_count"),
        avg("artist_popularity").alias("avg_artist_popularity"),
        avg("followers_total").alias("avg_artist_followers")
    )
    # 2. COALESCE after aggregation (data is much smaller now)
    .coalesce(16)
)

# 3. WRITE
track_artist_aggregates.write.mode("overwrite").parquet(f"{SILVER_BASE}/track_artist_aggregates")
print("✅ track_artist_aggregates written (16 partitions)")

# 4. VERIFY
print("\n📊 Sample data:")
spark.read.parquet(f"{SILVER_BASE}/track_artist_aggregates").show(5, truncate=False)

print(f"📈 Partition count: {spark.read.parquet(f'{SILVER_BASE}/track_artist_aggregates').rdd.getNumPartitions()}")

26/02/05 19:01:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 19:02:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 19:02:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 19:02:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 19:02:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 19:02:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 19:02:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 19:02:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 19:02:00 WARN RowBasedKeyValueBatch: Calling spill() on

✅ track_artist_aggregates written (16 partitions)

📊 Sample data:
+-----------+------------+---------------------+--------------------+
|track_rowid|artist_count|avg_artist_popularity|avg_artist_followers|
+-----------+------------+---------------------+--------------------+
|19         |1           |NULL                 |NULL                |
|26         |1           |NULL                 |NULL                |
|29         |1           |NULL                 |NULL                |
|54         |2           |NULL                 |NULL                |
|65         |1           |NULL                 |NULL                |
+-----------+------------+---------------------+--------------------+
only showing top 5 rows

📈 Partition count: 16


## 9. album_track_aggregates

In [6]:
album_track_aggregates = (
    tracks_clean
    # 1. GROUP BY
    .groupBy("album_rowid")
    .agg(
        count("*").alias("tracks_per_album"),
        avg("popularity").alias("avg_track_popularity"),
        sum("duration_seconds").alias("album_duration_total")
    )
    # 2. COALESCE (small result)
    .coalesce(8)
)

# 3. WRITE
album_track_aggregates.write.mode("overwrite").parquet(f"{SILVER_BASE}/album_track_aggregates")
print("✅ album_track_aggregates written (8 partitions)")

# 4. VERIFY
print("\n📊 Sample data:")
spark.read.parquet(f"{SILVER_BASE}/album_track_aggregates").show(5, truncate=False)

print(f"📈 Partition count: {spark.read.parquet(f'{SILVER_BASE}/album_track_aggregates').rdd.getNumPartitions()}")


✅ album_track_aggregates written (8 partitions)

📊 Sample data:
+-----------+----------------+--------------------+--------------------+
|album_rowid|tracks_per_album|avg_track_popularity|album_duration_total|
+-----------+----------------+--------------------+--------------------+
|50094312   |40              |0.0                 |7452.64             |
|50135990   |15              |0.0                 |2629.269            |
|50278095   |19              |0.0                 |8992.728            |
|50507966   |7               |0.0                 |1615.513            |
|50517657   |1               |0.0                 |187.392             |
+-----------+----------------+--------------------+--------------------+
only showing top 5 rows

📈 Partition count: 12


## 10. artist_collaboration_metrics

In [15]:
# OPTIMIZATION: Only process tracks with reasonable artist counts (≤ 4)
# This prevents explosion for compilation albums with 20+ artists
tracks_with_few_artists = (
    track_artist_expanded
    .groupBy("track_rowid")
    .agg(count("*").alias("artist_count"))
    .filter(col("artist_count") <= 4)  # Limit to avoid N² explosion
)

# Filter track_artist_expanded to reasonable tracks
track_artist_filtered = (
    track_artist_expanded
    .join(tracks_with_few_artists, "track_rowid", "inner")
    .select("track_rowid", "artist_rowid")
)

# Self-join to find pairs
ta = track_artist_filtered.alias("a")

pairs = (
    ta.join(
        track_artist_filtered.alias("b"), 
        "track_rowid"
    )
    .filter(col("a.artist_rowid") < col("b.artist_rowid"))  # Avoid duplicates (A-B same as B-A)
    .select(
        col("a.artist_rowid").alias("artist_rowid"),
        col("b.artist_rowid").alias("collaborator")
    )
)

artist_collaboration_metrics = (
    pairs
    .groupBy("artist_rowid")
    .agg(
        countDistinct("collaborator").alias("unique_collaborators"),
        count("*").alias("collaboration_count")
    )
    .coalesce(8)
)

# WRITE
artist_collaboration_metrics.write.mode("overwrite").parquet(f"{SILVER_BASE}/artist_collaboration_metrics")
print("✅ artist_collaboration_metrics written (8 partitions)")

# VERIFY
print("\n📊 Sample data:")
spark.read.parquet(f"{SILVER_BASE}/artist_collaboration_metrics").show(5, truncate=False)

print(f"📈 Partition count: {spark.read.parquet(f'{SILVER_BASE}/artist_collaboration_metrics').rdd.getNumPartitions()}")

26/02/05 19:16:46 WARN MemoryStore: Not enough space to cache rdd_181_8 in memory! (computed 3.8 MiB so far)
26/02/05 19:16:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 19:16:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 19:16:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 19:16:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 19:16:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 19:16:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 19:16:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 19:16:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBase

✅ artist_collaboration_metrics written (8 partitions)

📊 Sample data:
+------------+--------------------+-------------------+
|artist_rowid|unique_collaborators|collaboration_count|
+------------+--------------------+-------------------+
|3842776     |41                  |86                 |
|3323660     |34                  |755                |
|2659541     |5                   |11                 |
|1165983     |13                  |21                 |
|4452637     |28                  |69                 |
+------------+--------------------+-------------------+
only showing top 5 rows

📈 Partition count: 10


# GOLD

In [16]:
GOLD_BASE = "hdfs://localhost:9000/data_lake/gold"

In [3]:
# Core cleaned tables
artists_clean = spark.read.parquet(f"{SILVER_BASE}/artists_clean")
albums_clean = spark.read.parquet(f"{SILVER_BASE}/albums_clean")
tracks_clean = spark.read.parquet(f"{SILVER_BASE}/tracks_clean")

# Expanded relationship tables
track_artist_expanded = spark.read.parquet(f"{SILVER_BASE}/track_artist_expanded")
artist_album_expanded = spark.read.parquet(f"{SILVER_BASE}/artist_album_expanded")
artist_genre_expanded = spark.read.parquet(f"{SILVER_BASE}/artist_genre_expanded")
market_expanded = spark.read.parquet(f"{SILVER_BASE}/market_expanded")

# Aggregates
track_artist_aggregates = spark.read.parquet(f"{SILVER_BASE}/track_artist_aggregates")
album_track_aggregates = spark.read.parquet(f"{SILVER_BASE}/album_track_aggregates")
artist_collaboration_metrics = spark.read.parquet(f"{SILVER_BASE}/artist_collaboration_metrics")

In [4]:
# # Core cleaned tables
# artists_clean = spark.read.parquet(f"{SILVER_BASE}/artists_clean").cache()
# albums_clean = spark.read.parquet(f"{SILVER_BASE}/albums_clean").cache()
# tracks_clean = spark.read.parquet(f"{SILVER_BASE}/tracks_clean").cache()

# # Expanded relationship tables
# track_artist_expanded = spark.read.parquet(f"{SILVER_BASE}/track_artist_expanded").cache()
# artist_album_expanded = spark.read.parquet(f"{SILVER_BASE}/artist_album_expanded").cache()
# artist_genre_expanded = spark.read.parquet(f"{SILVER_BASE}/artist_genre_expanded").cache()
# market_expanded = spark.read.parquet(f"{SILVER_BASE}/market_expanded").cache()

# # Aggregates
# track_artist_aggregates = spark.read.parquet(f"{SILVER_BASE}/track_artist_aggregates").cache()
# album_track_aggregates = spark.read.parquet(f"{SILVER_BASE}/album_track_aggregates").cache()
# artist_collaboration_metrics = spark.read.parquet(f"{SILVER_BASE}/artist_collaboration_metrics").cache()

# # Force caching
# for df in [artists_clean, albums_clean, tracks_clean, track_artist_expanded, 
#            artist_album_expanded, artist_genre_expanded, market_expanded,
#            track_artist_aggregates, album_track_aggregates, artist_collaboration_metrics]:
#     df.count()

# print("✅ All Silver tables cached")

# 1. track_performance

In [ ]:
# Calculate market counts efficiently
market_counts = (
    market_expanded
    .groupBy("track_rowid")
    .agg(countDistinct("market_code").alias("market_count"))
)

track_performance = (
    tracks_clean
    # Join 1: Album release context (cached albums_clean)
    .join(
        albums_clean.select("album_rowid", "release_year"), 
        "album_rowid", 
        "left"
    )
    
    # Join 2: Artist aggregates (cached)
    .join(track_artist_aggregates, "track_rowid", "left")
    
    # Join 3: Market reach
    .join(market_counts, "track_rowid", "left")
    
    # Derive track age
    .withColumn(
        "track_age_days",
        datediff(current_date(), col("fetched_ts"))
    )
    
    # Fill nulls from left joins
    .fillna(0, subset=["market_count", "artist_count", "avg_artist_popularity", "avg_artist_followers"])
    
    # Repartition for optimal write (large table)
    .repartition(128, "track_rowid")
)

# WRITE
track_performance.write.mode("overwrite").parquet(f"{GOLD_BASE}/track_performance")
print("✅ track_performance written (128 partitions)")

# VERIFY
print("\n📊 Sample data:")
spark.read.parquet(f"{GOLD_BASE}/track_performance").show(5, truncate=False)

print(f"📈 Partition count: {spark.read.parquet(f'{GOLD_BASE}/track_performance').rdd.getNumPartitions()}")


26/02/05 22:52:40 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 22:52:40 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 22:52:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 22:52:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 22:53:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 22:53:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 22:53:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 22:54:04 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/05 22:54:15 WARN RowBasedKeyValueBatch: Calling spill() on

# 2. artist_performance

In [ ]:
artist_tracks = (
    track_artist_expanded
    .groupBy("artist_rowid")
    .agg(countDistinct("track_rowid").alias("total_tracks"))
)

# Aggregate 2: Total albums per artist
artist_albums_agg = (
    artist_album_expanded
    .groupBy("artist_rowid")
    .agg(countDistinct("album_rowid").alias("total_albums"))
)

# Aggregate 3: Genre count per artist
artist_genres_agg = (
    artist_genre_expanded
    .groupBy("artist_rowid")
    .agg(countDistinct("genre").alias("genre_count"))
)

# Join all metrics together
artist_performance = (
    artists_clean
    .join(artist_tracks, "artist_rowid", "left")
    .join(artist_albums_agg, "artist_rowid", "left")
    .join(artist_genres_agg, "artist_rowid", "left")
    .join(artist_collaboration_metrics, "artist_rowid", "left")
    
    # Fill nulls
    .fillna(0)
    
    # Coalesce (result is smaller than inputs)
    .repartition(16, "artist_rowid")
)

# WRITE
artist_performance.write.mode("overwrite").parquet(f"{GOLD_BASE}/artist_performance")
print("✅ artist_performance written (16 partitions)")

# VERIFY
print("\n📊 Sample data:")
spark.read.parquet(f"{GOLD_BASE}/artist_performance").show(5, truncate=False)

print(f"📈 Partition count: {spark.read.parquet(f'{GOLD_BASE}/artist_performance').rdd.getNumPartitions()}")


# 3. genre_intelligence

In [ ]:

# Aggregate 1: Track metrics per genre
genre_tracks = (
    track_artist_expanded
    .join(artist_genre_expanded, "artist_rowid")
    .join(tracks_clean, "track_rowid")
    .groupBy("genre")
    .agg(
        countDistinct("track_rowid").alias("track_count"),
        avg("popularity").alias("avg_track_popularity")
    )
)

# Aggregate 2: Market spread per genre
genre_markets = (
    track_artist_expanded
    .join(artist_genre_expanded, "artist_rowid")
    .join(market_expanded, "track_rowid")
    .groupBy("genre")
    .agg(countDistinct("market_code").alias("market_spread"))
)

# Join genre metrics
genre_intelligence = (
    genre_tracks
    .join(genre_markets, "genre", "left")
    .fillna(0)
    .coalesce(4)  # Small table
)

# WRITE
genre_intelligence.write.mode("overwrite").parquet(f"{GOLD_BASE}/genre_intelligence")
print("✅ genre_intelligence written (4 partitions)")

# VERIFY
print("\n📊 Sample data:")
spark.read.parquet(f"{GOLD_BASE}/genre_intelligence").show(5, truncate=False)

print(f"📈 Partition count: {spark.read.parquet(f'{GOLD_BASE}/genre_intelligence').rdd.getNumPartitions()}")


# 4. collaboration_network_summary

In [ ]:

collaboration_network_summary = (
    artists_clean
    .join(artist_collaboration_metrics, "artist_rowid", "left")
    .fillna(0)
    .repartition(16, "artist_rowid")
)

# WRITE
collaboration_network_summary.write.mode("overwrite").parquet(f"{GOLD_BASE}/collaboration_network_summary")
print("✅ collaboration_network_summary written (16 partitions)")

# VERIFY
print("\n📊 Sample data:")
spark.read.parquet(f"{GOLD_BASE}/collaboration_network_summary").show(5, truncate=False)

print(f"📈 Partition count: {spark.read.parquet(f'{GOLD_BASE}/collaboration_network_summary').rdd.getNumPartitions()}")


# 5. track_success_features

In [ ]:
track_success_features = (
    tracks_clean
    # Join 1: Artist aggregates
    .join(track_artist_aggregates, "track_rowid", "left")
    
    # Join 2: Album release year
    .join(
        albums_clean.select("album_rowid", "release_year"), 
        "album_rowid", 
        "left"
    )
    
    # Join 3: Market counts
    .join(market_counts, "track_rowid", "left")
    
    # Fill nulls
    .fillna(0)
    
    # Repartition for large table
    .repartition(128, "track_rowid")
)

# WRITE
track_success_features.write.mode("overwrite").parquet(f"{GOLD_BASE}/track_success_features")
print("✅ track_success_features written (128 partitions)")

# VERIFY
print("\n📊 Sample data:")
spark.read.parquet(f"{GOLD_BASE}/track_success_features").show(5, truncate=False)

print(f"📈 Partition count: {spark.read.parquet(f'{GOLD_BASE}/track_success_features').rdd.getNumPartitions()}")


# 6. release_trends

In [ ]:
# Albums per year
album_releases = (
    albums_clean
    .groupBy("release_year")
    .agg(countDistinct("album_rowid").alias("album_releases"))
)

# Tracks per year (via album join)
track_releases = (
    tracks_clean
    .join(albums_clean.select("album_rowid", "release_year"), "album_rowid")
    .groupBy("release_year")
    .agg(countDistinct("track_rowid").alias("track_releases"))
)

release_trends = (
    album_releases
    .join(track_releases, "release_year", "left")
    .orderBy("release_year")
    .coalesce(2)  # Very small table
)

# WRITE
release_trends.write.mode("overwrite").parquet(f"{GOLD_BASE}/release_trends")
print("✅ release_trends written (2 partitions)")

# VERIFY
print("\n📊 Sample data:")
spark.read.parquet(f"{GOLD_BASE}/release_trends").show(10, truncate=False)

print(f"📈 Partition count: {spark.read.parquet(f'{GOLD_BASE}/release_trends').rdd.getNumPartitions()}")
